In [1]:
# Part 1 - Create the Project List (Test Type Detection Only)
#========= create Project_List basic ====================

import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Project_List.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                  'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other': ['instrumentation', 'instrument']
}

# === STRUCTURE TO HOLD RESULTS ===
project_results = {}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

# === MAIN PARSER ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            detected = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': detected, 'error': True}
            return {'types': detected, 'error': False}
    except Exception:
        return {'types': set(), 'error': True}

# === PROJECT SCANNER ===
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {'types': set(), 'errors': 0}
            project_results[project_name]['types'].update(result['types'])
            if result['error']:
                project_results[project_name]['errors'] += 1

# === EXPORT CSV ===
rows = []
for project, result in project_results.items():
    cleaned_types = result['types']
    if 'Other' in cleaned_types and len(cleaned_types) > 1:
        cleaned_types = cleaned_types - {'Other'}

    rows.append({
        'project': project,
        'test_types': ', '.join(sorted(cleaned_types)) if cleaned_types else 'none',
        'yaml_errors': result['errors']
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
Project_List = df.copy()

print(f"\n✅ Summary written to: {OUTPUT_CSV}")



✅ Summary written to: C:\GitHub\Android-Mobile-Apps\Project_List.csv


In [1]:
# ========== PART 1: Full Project Analysis ==========

import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\latest_config_files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV_1 = os.path.join(OUTPUT_DIR, "Project_List_Only.csv")

# === REUSED MAIN PARSER FOR CONSISTENT YAML ERROR DETECTION ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            content = yaml.safe_load(raw)
            if not content:
                return {'error': True}
            return {'error': False}
    except Exception:
        return {'error': True}

# === PARSE YAML FILES AND COUNT ERRORS ===
project_results = {}
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {'project': project_name, 'yaml_errors': 0}

            if result['error']:
                project_results[project_name]['yaml_errors'] += 1

# === BUILD FINAL DATAFRAME ===
df = pd.DataFrame(list(project_results.values())).sort_values(by='project')

# === EXPORT TO CSV ===
df.to_csv(OUTPUT_CSV_1, index=False)
print(f"\n✅ Project List-Part 1 written to: {OUTPUT_CSV_1}")



✅ Project List-Part 1 written to: C:\GitHub\Android-Mobile-Apps\Project_List_Only.csv
